In [ ]:
import pandas as pd
import openpyxl
import os
import re
from datetime import datetime
import win32com.client as win32
import win32api

# ==========================
# CONFIGURAÇÕES
# ==========================

CAMINHO_BASE   = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\TESTE\Base_Extracao_PE_Produto_Foco_TESTE1.xlsx'
PASTA_SAIDA    = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\Arquivos_Gerados'
ARQUIVO_LOG    = r'C:\Users\v.alves\OneDrive - SPOT\Área de Trabalho\ENVIOS PRODUTO FOCO\LOG_ENVIO_PE.xlsx'

ABA_DADOS      = 'BASE_PE'
ABA_BASE       = 'BASE_PE'
ABA_PIVOT      = 'TT_PE'

# Colunas A até O (15 colunas)
COLUNAS_MODELO = [
    'COD_PESQUISA', 'DATA', 'COD_LOJA', 'COD_SAP', 'NOME_FANTASIA',
    'BANDEIRA', 'DES_REGIAO', 'NOM_PESSOA_COMPLETO', 'DES_CATEGORIA',
    'DES_SUB_CATEGORIA', 'DES_TIPO_PONTO_EXTRA', 'FOTO', 'CHAVE',
    'EXISTE', 'PE_RETORNO'
]

EMAILS_CC      = [
    'l.basaia@spotpromo.com.br'
]

MAX_TENTATIVAS = 2
MODO_TESTE     = False


# ==========================
# HELPERS
# ==========================

def tipo_label(modo_teste):
    return "TESTE" if modo_teste else "PRODUÇÃO"

def normalizar_df(df):
    """Normaliza colunas de timestamp com fuso horário para evitar erros no Excel."""
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            if df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_localize(None)
    return df

def limpar_nome(nome):
    return re.sub(r'[\\/*?:"<>|]', "", str(nome))

def path_curto(caminho):
    return win32api.GetShortPathName(os.path.abspath(caminho))


# ==========================
# PREPARAÇÃO
# ==========================

df = pd.read_excel(CAMINHO_BASE, sheet_name=ABA_DADOS)
os.makedirs(PASTA_SAIDA, exist_ok=True)

data_log  = datetime.now().strftime('%d-%m-%Y %H:%M:%S')
data_nome = datetime.now().strftime('%d-%m-%Y')


# ==========================
# VALIDAÇÃO
# ==========================

def validar_base():
    ausentes = [c for c in ['ESPECIALISTA', 'EMAIL ESPECIALISTA', 'GERENTE', 'EMAIL GERENTE']
                if c not in df.columns]
    if ausentes:
        raise ValueError(f"Colunas ausentes na base: {ausentes}")
    print("✅ Base validada com sucesso.")


# ==========================
# LOG
# ==========================

registros_log = []

def registrar_log(tipo, nome, email, status, erro=""):
    registros_log.append({
        "DATA_ENVIO": data_log, "TIPO": tipo, "NOME": nome,
        "EMAIL": email, "STATUS": status, "ERRO": erro
    })

def salvar_log():
    if not registros_log:
        return
    novo = pd.DataFrame(registros_log)
    if os.path.exists(ARQUIVO_LOG):
        final = pd.concat([pd.read_excel(ARQUIVO_LOG), novo], ignore_index=True)
    else:
        final = novo
    final.to_excel(ARQUIVO_LOG, index=False)
    print(f"\n📋 Log salvo: {len(registros_log)} registros em {ARQUIVO_LOG}")


# ==========================
# GERAR RELATÓRIO
# ==========================

def gerar_relatorio(df_filtrado, nome_saida):
    caminho_final     = os.path.abspath(nome_saida)
    colunas_presentes = [c for c in COLUNAS_MODELO if c in df_filtrado.columns]
    df_para_gravar    = normalizar_df(df_filtrado[colunas_presentes].reset_index(drop=True))

    print(f"      → gravando {len(df_para_gravar)} linhas | {os.path.basename(nome_saida)}")

    # ── TT_PE: resumo de QTD_PE por SUB_CATEGORIA e TIPO_PONTO_EXTRA ──
    cols_group = ['DES_SUB_CATEGORIA', 'DES_TIPO_PONTO_EXTRA']
    col_chave  = 'CHAVE'
    cols_disp  = [c for c in cols_group if c in df_para_gravar.columns]

    resumo = None
    if cols_disp and col_chave in df_para_gravar.columns:
        resumo = (
            df_para_gravar
            .groupby(cols_disp, dropna=False)[col_chave]
            .nunique()
            .reset_index()
            .rename(columns={col_chave: 'QTD_PE'})
            .sort_values(cols_disp)
            .reset_index(drop=True)
        )
        total_pe = int(resumo['QTD_PE'].sum())

        # Adiciona linha de total
        linha_total = pd.DataFrame(
            [[''] * (len(cols_disp) - 1) + ['TOTAL', total_pe]],
            columns=cols_disp + ['QTD_PE']
        )
        resumo = pd.concat([resumo, linha_total], ignore_index=True)

    # ── Escreve o arquivo usando pandas ExcelWriter (muito mais rápido) ──
    with pd.ExcelWriter(caminho_final, engine='openpyxl') as writer:
        df_para_gravar.to_excel(writer, sheet_name=ABA_BASE,  index=False)
        if resumo is not None:
            resumo.to_excel(writer, sheet_name=ABA_PIVOT, index=False)
            print(f"      → TT_PE: {len(resumo) - 1} combinações | TOTAL PE: {total_pe}")
        else:
            # Cria aba TT_PE vazia se não houver colunas suficientes
            pd.DataFrame().to_excel(writer, sheet_name=ABA_PIVOT, index=False)


# ==========================
# ENVIAR E-MAIL
# ==========================

def enviar_email(destinatario, nome, arquivo, tipo, outlook_app):

    if pd.isna(destinatario) or str(destinatario).strip() == "":
        registrar_log(tipo, nome, destinatario, "ERRO", "Sem e-mail")
        return

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            mail = outlook_app.CreateItem(0)

            mail.To = str(destinatario).strip()
            mail.CC = "; ".join(EMAILS_CC)
            mail.Subject = f"Relatório Semanal - Pontos Extras ({data_nome})"

            link_dashboard = "https://tigre.spotpromo.com.br/sgdm/pages.php?url=db03ad52-3e36-4d52-b3b5-e64bb9289fa1"

            # 🔹 Abre o e-mail para carregar assinatura automática
            mail.Display()
            assinatura = mail.HTMLBody

            # 🔹 Corpo do e-mail
            corpo_html = f"""
            <html>
            <body style="font-family: Arial; font-size: 12pt;">

                <p>Olá {nome},</p>

                <p>
                Segue em anexo o relatório atualizado de 
                <b>Pontos Extras Relacionados aos Produtos Foco</b>.
                </p>

                <p>
                <b>OBS:</b> A coluna CHAVE_TI é utilizada para diferenciarmos 
                se o Ponto Extra é referente a retorno ou uma nova aquisição.
                </p>

                <p>
                Para uma análise mais completa, acesse o dashboard:
                </p>

                <p>
                <a href="{link_dashboard}"
                style="background-color:#0078D4; color:white; padding:10px 15px; text-decoration:none; border-radius:5px;">
                📊 Acessar Dashboard
                </a>
                </p>

                <br>

                <p>Qualquer dúvida fico à disposição.</p>

                <p>Att,<p>

            </body>
            </html>
            """

            # 🔹 Junta corpo + assinatura
            mail.HTMLBody = corpo_html + assinatura

            # 🔹 Anexo
            mail.Attachments.Add(os.path.abspath(arquivo))

            # 🔹 Envio
            mail.Send()

            registrar_log(tipo, nome, destinatario, "ENVIADO")
            break

        except Exception as e:
            if tentativa == MAX_TENTATIVAS:
                registrar_log(tipo, nome, destinatario, "ERRO", str(e))
            else:
                continue

# ==========================
# VALIDAÇÕES INICIAIS
# ==========================

validar_base()

print(f"Base carregada: {len(df)} registros | "
      f"{df['ESPECIALISTA'].nunique()} especialistas | "
      f"{df['GERENTE'].nunique()} gerentes")

if not MODO_TESTE:
    total_dest = df['ESPECIALISTA'].nunique() + df['GERENTE'].nunique()
    resposta = input(
        f"\n⚠️  MODO PRODUÇÃO — {total_dest} destinatários. Confirma? [s/n]: "
    ).strip().lower()
    if resposta != 's':
        print("Cancelado.")
        raise SystemExit(0)

if MODO_TESTE:
    print("\n⚠️  MODO TESTE — nenhum e-mail será enviado.\n")
    PASTA_ATUAL = os.path.join(PASTA_SAIDA, 'TESTE')
    os.makedirs(PASTA_ATUAL, exist_ok=True)
else:
    PASTA_ATUAL = PASTA_SAIDA

outlook_app = None if MODO_TESTE else win32.Dispatch('outlook.application')

try:
    # --- ESPECIALISTAS ---
    especialistas = df['ESPECIALISTA'].dropna().unique()
    print(f"Processando {len(especialistas)} especialistas...")

    for especialista in especialistas:
        df_esp       = df[df['ESPECIALISTA'] == especialista].copy()
        email        = df_esp['EMAIL ESPECIALISTA'].iloc[0]
        nome_limpo   = limpar_nome(especialista)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Especialista_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {especialista} → {len(df_esp)} registros")
        try:
            gerar_relatorio(df_esp, nome_arquivo)
            if MODO_TESTE:
                registrar_log("Especialista", especialista, email, "GERADO")
                print(f"    ✅ {nome_arquivo}")
            else:
                enviar_email(email, especialista, nome_arquivo, "Especialista", outlook_app)
        except Exception as e:
            registrar_log("Especialista", especialista, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")

    # --- GERENTES ---
    gerentes = df['GERENTE'].dropna().unique()
    print(f"\nProcessando {len(gerentes)} gerentes...")

    for gerente in gerentes:
        df_ger       = df[df['GERENTE'] == gerente].copy()
        email        = df_ger['EMAIL GERENTE'].iloc[0]
        nome_limpo   = limpar_nome(gerente)
        prefixo      = "TESTE_" if MODO_TESTE else ""
        nome_arquivo = os.path.join(PASTA_ATUAL, f"{prefixo}Gerente_{nome_limpo}_{data_nome}.xlsx")

        print(f"  [{tipo_label(MODO_TESTE)}] {gerente} → {len(df_ger)} registros")
        try:
            gerar_relatorio(df_ger, nome_arquivo)
            if MODO_TESTE:
                registrar_log("Gerente", gerente, email, "GERADO")
                print(f"    ✅ {nome_arquivo}")
            else:
                enviar_email(email, gerente, nome_arquivo, "Gerente", outlook_app)
        except Exception as e:
            registrar_log("Gerente", gerente, email, "ERRO", str(e))
            print(f"    ❌ ERRO: {e}")

finally:
    salvar_log()

if MODO_TESTE:
    print(f"\n✅ Modo teste concluído! Arquivos em: {PASTA_ATUAL}")
else:
    print("\n✅ Processo finalizado com sucesso!")

✅ Base validada com sucesso.
Base carregada: 258 registros | 4 especialistas | 3 gerentes
Processando 4 especialistas...
  [PRODUÇÃO] GUSTAVO NEVES → 61 registros
      → gravando 61 linhas | Especialista_GUSTAVO NEVES_19-03-2026.xlsx
      → TT_PE: 15 combinações | TOTAL PE: 61
  [PRODUÇÃO] KIMBERLEY MIRANDA → 63 registros
      → gravando 63 linhas | Especialista_KIMBERLEY MIRANDA_19-03-2026.xlsx
      → TT_PE: 17 combinações | TOTAL PE: 63
  [PRODUÇÃO] JONATAS PEQUENO → 60 registros
      → gravando 60 linhas | Especialista_JONATAS PEQUENO_19-03-2026.xlsx
      → TT_PE: 16 combinações | TOTAL PE: 59
  [PRODUÇÃO] RODRIGO DELMONDES → 74 registros
      → gravando 74 linhas | Especialista_RODRIGO DELMONDES_19-03-2026.xlsx
      → TT_PE: 17 combinações | TOTAL PE: 72

Processando 3 gerentes...
  [PRODUÇÃO] EDFLANKLIN → 52 registros
      → gravando 52 linhas | Gerente_EDFLANKLIN_19-03-2026.xlsx
      → TT_PE: 15 combinações | TOTAL PE: 43
  [PRODUÇÃO] DIOGO PESSOA → 113 registros
      